In [1]:
import numpy as np
import pandas as pd

# Configurar semilla para reproducibilidad
np.random.seed(42)
n_registros = 2500

# 1. GENERACIÓN DEL DATASET PRINCIPAL: df_ventas_giga
clientes = [
    "TechCorp",
    "AlphaLabs",
    "BetaOmega",
    "GlobalNet",
    "DeltaSystems",
    "NovaTech",
    "QuantumSoft",
]
planes = ["Basico", "Familiar", "Premium"]
regiones = ["Norte", "Sur", "Este", "Oeste", None]  # Incluye nulos intencionales

df_ventas_giga = pd.DataFrame(
    {
        "ID_Pedido": range(10001, 10001 + n_registros),
        "Cliente": np.random.choice(clientes, n_registros),
        "Plan_Suscripcion": np.random.choice(planes, n_registros, p=[0.5, 0.3, 0.2]),
        "Compras_2026": np.random.randint(1, 30, n_registros).astype(float),
        "Total_Gastado_USD": np.random.uniform(50, 2500, n_registros),
        "Region": np.random.choice(regiones, n_registros, p=[0.3, 0.2, 0.2, 0.2, 0.1]),
    }
)

# Inyectar imperfecciones (Espacios, Nulos y Duplicados)
# Espacios invisibles en clientes
df_ventas_giga.loc[df_ventas_giga["Cliente"] == "TechCorp", "Cliente"] = (
    " TechCorp "
)
# Nulos en Compras_2026
df_ventas_giga.loc[
    np.random.choice(df_ventas_giga.index, 120, replace=False), "Compras_2026"
] = np.nan
# Filas duplicadas idénticas
duplicados = df_ventas_giga.sample(n=45, random_state=42)
df_ventas_giga = pd.concat([df_ventas_giga, duplicados], ignore_index=True)


# 2. GENERACIÓN DEL DATASET DE LOGÍSTICA: df_envios_giga
# Solo el 90% de los pedidos tienen registro de envío
pedidos_con_envio = df_ventas_giga["ID_Pedido"].unique()
pedidos_con_envio = np.random.choice(
    pedidos_con_envio, int(len(pedidos_con_envio) * 0.9), replace=False
)

df_envios_giga = pd.DataFrame(
    {
        "ID_Pedido": pedidos_con_envio,
        "Dias_Entrega": np.random.randint(1, 15, len(pedidos_con_envio)),
        "Estado_Envio": np.random.choice(
            ["Entregado", "Retrasado", "En Camino"],
            len(pedidos_con_envio),
            p=[0.7, 0.15, 0.15],
        ),
    }
)

print(f"Dataset df_ventas_giga creado con {df_ventas_giga.shape[0]} registros.")
print(f"Dataset df_envios_giga creado con {df_envios_giga.shape[0]} registros.")


Dataset df_ventas_giga creado con 2545 registros.
Dataset df_envios_giga creado con 2250 registros.


In [2]:
df_ventas_giga.head(5)

,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region
0,10001,QuantumSoft,Familiar,20.0,1177.870664,NaN
1,10002,GlobalNet,Basico,22.0,1788.694792,Sur
2,10003,DeltaSystems,Basico,16.0,2460.093379,Sur
3,10004,QuantumSoft,Premium,23.0,172.929608,NaN
4,10005,BetaOmega,Premium,11.0,286.901671,Norte


In [3]:
df_envios_giga.head(5)

,ID_Pedido,Dias_Entrega,Estado_Envio
0,10189,2,Entregado
1,11258,11,Entregado
2,10672,1,Entregado
3,10378,13,En Camino
4,11168,14,Entregado


# 📊 CASO DE NEGOCIO: Auditoría de Datos a Gran Escala (Giga-Ventas)

## 🛠️ Desafío 1 (Nivel: Fácil) - Fase de Limpieza de Texto
* **Contexto:** El Director Comercial nota que los filtros de clientes no cuadran. Al revisar los datos masivos, el cliente `"TechCorp"` se encuentra fragmentado en los reportes debido a espacios en blanco accidentales introducidos por el sistema de captura.
* **Objetivo:** Detectar cuántas filas exactas contienen el nombre del cliente con espacios en blanco invisibles a los lados en `df_ventas_giga`, y luego corregir todo el dataset eliminando dichos espacios de forma definitiva.
* **Pista conceptual:** Puedes contar cuántas filas cumplen con tener espacios antes o después de la palabra usando operadores de cadenas, y luego reescribir la columna con el método vectorizado de strings adecuado.


In [4]:
#Consultar los espacios en blanco
df_ventas_giga[df_ventas_giga['Cliente'].str.contains(r"\s", na=False)]


,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region
25,10026,TechCorp,Familiar,5.0,2240.137101,Norte
31,10032,TechCorp,Premium,2.0,1159.420773,Este
32,10033,TechCorp,Familiar,13.0,732.801074,Norte
48,10049,TechCorp,Familiar,20.0,263.069200,Norte
54,10055,TechCorp,Familiar,11.0,2278.508708,Sur
...,...,...,...,...,...,...
2491,12492,TechCorp,Basico,5.0,113.068904,Sur
2499,12500,TechCorp,Premium,26.0,81.875023,Oeste
2502,11065,TechCorp,Premium,20.0,1510.833509,Sur
2503,12288,TechCorp,Basico,13.0,1378.634940,Oeste


In [5]:
#Llevar a un dataframe lo que voy a eliminar
df_espacios_nombre = df_ventas_giga[df_ventas_giga['Cliente'].str.contains(r"\s", na=False)]

In [6]:
#Mostrar el dataframe en donde se guardó la información
df_espacios_nombre

,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region
25,10026,TechCorp,Familiar,5.0,2240.137101,Norte
31,10032,TechCorp,Premium,2.0,1159.420773,Este
32,10033,TechCorp,Familiar,13.0,732.801074,Norte
48,10049,TechCorp,Familiar,20.0,263.069200,Norte
54,10055,TechCorp,Familiar,11.0,2278.508708,Sur
...,...,...,...,...,...,...
2491,12492,TechCorp,Basico,5.0,113.068904,Sur
2499,12500,TechCorp,Premium,26.0,81.875023,Oeste
2502,11065,TechCorp,Premium,20.0,1510.833509,Sur
2503,12288,TechCorp,Basico,13.0,1378.634940,Oeste


## 🔍 Desafío 2 (Nivel: Medio) - Fase de Auditoría y Duplicados Complejos
* **Contexto:** Al consolidar miles de registros, es común que el sistema registre la misma transacción más de una vez por errores de red. Sin embargo, no todos los registros repetidos son errores: un cliente puede realizar compras legítimas por el mismo monto en el mismo año.
* **Objetivo:** Identificar cuántas filas duplicadas exactas (idénticas en todas sus columnas) existen en `df_ventas_giga`, aislar esos registros duplicados en un nuevo DataFrame llamado `df_giga_duplicados` para enviarlo a sistemas como evidencia, y finalmente eliminarlos del DataFrame original manteniendo solo la primera ocurrencia.
* **Pista conceptual:** Explora el uso de los métodos de detección de duplicados configurando el parámetro que te permite marcar cuáles filas considerar como duplicadas, y luego aplica la remoción permanente.


In [7]:
#Consultar duplicados
df_ventas_giga[df_ventas_giga.duplicated()]


,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region
2500,11448,AlphaLabs,Basico,17.0,597.826243,Norte
2501,11115,QuantumSoft,Premium,19.0,1129.477629,Norte
2502,11065,TechCorp,Premium,20.0,1510.833509,Sur
2503,12288,TechCorp,Basico,13.0,1378.634940,Oeste
2504,11538,GlobalNet,Basico,7.0,1324.792884,Oeste
2505,10669,BetaOmega,Basico,4.0,1985.703962,Este
2506,11584,GlobalNet,Familiar,3.0,490.145484,Oeste
2507,12405,GlobalNet,Basico,23.0,800.286151,Norte
2508,10498,GlobalNet,Basico,11.0,1042.732999,NaN
2509,12481,DeltaSystems,Basico,19.0,1665.586955,Oeste


In [8]:
#Llevar una copia a un nuevo dataframe
df_giga_duplicados = df_ventas_giga[df_ventas_giga.duplicated()]

In [9]:
#Mostrar el dataframe en donde se guardó la información
df_giga_duplicados.head(5)

,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region
2500,11448,AlphaLabs,Basico,17.0,597.826243,Norte
2501,11115,QuantumSoft,Premium,19.0,1129.477629,Norte
2502,11065,TechCorp,Premium,20.0,1510.833509,Sur
2503,12288,TechCorp,Basico,13.0,1378.634940,Oeste
2504,11538,GlobalNet,Basico,7.0,1324.792884,Oeste


In [10]:
#Eliminar del dataframe original los registros duplicados
df_ventas_giga = df_ventas_giga.drop_duplicates()

In [11]:
#Mostrar el dataframe depurado
df_ventas_giga.head(5)

,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region
0,10001,QuantumSoft,Familiar,20.0,1177.870664,NaN
1,10002,GlobalNet,Basico,22.0,1788.694792,Sur
2,10003,DeltaSystems,Basico,16.0,2460.093379,Sur
3,10004,QuantumSoft,Premium,23.0,172.929608,NaN
4,10005,BetaOmega,Premium,11.0,286.901671,Norte


## 🌋 Desafío 3 (Nivel: Difícil) - Imputación Condicional Avanzada por Segmentos
* **Contexto:** El Director Comercial rechaza la idea de rellenar los valores vacíos de las compras usando el promedio global de toda la empresa. Nos explica que los clientes del plan "Premium" compran muchísimo más que los del plan "Basico". Si usamos un promedio general, inflaremos artificialmente las compras de los clientes básicos y reduciremos drásticamente las de los premium, distorsionando el comportamiento real del negocio.
* **Objetivo:** Calcular el promedio de la columna `Compras_2026` de forma separada para cada tipo de `Plan_Suscripcion` (Basico, Familiar, Premium). Después, imputar (rellenar) los valores nulos (`NaN`) de la columna `Compras_2026` utilizando el promedio específico del plan al que pertenece ese cliente en esa fila, repitiendo el proceso para cada plan con filtros avanzados.
* **Pista conceptual:** Puedes resolver este reto en tres bloques independientes (uno para cada plan). Para cada uno, primero calcula la media filtrando solo ese plan. Luego, usando el indexador `.loc` junto con una condición doble (operador `&`), localiza las filas que correspondan a ese plan específico **y** que tengan valores nulos (`.isnull()`) en `Compras_2026` para asignarles el promedio calculado.


In [12]:
#Calcular la media para cada plan
media_basico = df_ventas_giga[df_ventas_giga['Plan_Suscripcion']== 'Basico']['Compras_2026'].mean()
media_familiar = df_ventas_giga[df_ventas_giga['Plan_Suscripcion']== 'Familiar']['Compras_2026'].mean()
media_premium = df_ventas_giga[df_ventas_giga['Plan_Suscripcion']== 'Premium']['Compras_2026'].mean()

#Filtrar los vacíos y asignar la media según el plan

#Basico
df_ventas_giga.loc[(df_ventas_giga['Plan_Suscripcion'] == 'Basico') & 
                   (df_ventas_giga['Compras_2026'].isnull()), 
                   'Compras_2026'] = media_basico

#Familiar
df_ventas_giga.loc[(df_ventas_giga['Plan_Suscripcion'] == 'Familiar') & 
                   (df_ventas_giga['Compras_2026'].isnull()), 
                   'Compras_2026'] = media_familiar

#Premium
df_ventas_giga.loc[(df_ventas_giga['Plan_Suscripcion'] == 'Premium') & 
                   (df_ventas_giga['Compras_2026'].isnull()), 
                   'Compras_2026'] = media_premium

In [13]:
#Visualizar el dataframe si hay Compras en blanco
df_ventas_giga[(df_ventas_giga['Compras_2026'].isna()) | (df_ventas_giga['Compras_2026'].isnull())]

,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region


# 📊 ETAPA 2: Consolidación y Reportes de Alto Impacto (Giga-Ventas)

## 🛠️ Desafío 4 (Nivel: Fácil) - Consolidación por Pérdida Cero
* **Contexto:** Logística nos informa que el dataset `df_envios_giga` solo contiene datos de los pedidos que ya salieron del almacén. El Director Comercial quiere un reporte consolidado, pero nos advierte que bajo ninguna circunstancia podemos borrar o ignorar los pedidos que aún no se han enviado, ya que representan ingresos reales que deben aparecer en la auditoría.
* **Objetivo:** Unir horizontalmente `df_ventas_giga` con `df_envios_giga` usando la columna clave común, asegurando que todos los registros de ventas se mantengan intactos en un nuevo DataFrame llamado `df_consolidado_giga`.
* **Pista conceptual:** Utiliza el método de combinación de tablas seleccionando el tipo de unión (*join*) que prioriza la conservación absoluta de la tabla declarada a la izquierda de la operación.


<details>
  <summary>▶️ Haz clic aquí para ver la pistas de inner, left y right</summary>
  1. ¿Cuándo usar how='left'? (El Salvador de Información)
• La regla de oro: "Mantén todo lo de mi tabla principal, sin importar qué".
• Cuándo usarlo: Cuando tienes una tabla primaria (como tus clientes o tus ventas) y quieres pegarle información secundaria (como sus envíos o sus comentarios), pero no quieres que se borre ninguna venta si el cliente aún no tiene envíos o comentarios asignados.
• Resultado: La tabla final tendrá exactamente el mismo tamaño que tu tabla de la izquierda. Lo que no se encuentre en la derecha se rellena con NaN.
2. ¿Cuándo usar how='right'? (El Espejo del Left)
• La regla de oro: "Mantén todo lo de la tabla secundaria que está a la derecha".
• Cuándo usarlo: Funciona exactamente igual que el left, pero prioriza la tabla que pones en segundo lugar dentro del código.
• Consejo profesional: En la práctica real, casi no se usa right. Los científicos de datos prefieren voltear el orden de las tablas en el código y seguir usando left porque es más natural para los seres humanos leer de izquierda a derecha.
3. ¿Cuándo usar how='inner'? (La Intersección Estricta)
• La regla de oro: "Solo quiero lo que esté presente en AMBAS tablas al mismo tiempo".
• Cuándo usarlo: Cuando necesitas una coincidencia perfecta y obligatoria. Por ejemplo, si tienes una lista de alumnos y una lista de exámenes aprobados, y el director te pide un reporte únicamente de los alumnos que presentaron examen. Si un alumno no hizo el examen, el inner lo borra del reporte final automáticamente.
• Resultado: Tu tabla final suele ser más pequeña que cualquiera de las dos originales, porque elimina los registros huérfanos de ambos lados.
📊 Resumen para recordar siempre:
• ¿Tengo una tabla principal que no se puede encoger? → left
• ¿Solo me importan las filas que coincidan perfectamente en ambos lados? → inner
Ahora que revisamos estas diferencias, cuéntame:
• ¿Te hace sentido por qué el ejercicio anterior requería un left en lugar de un inner?
</details>



In [14]:
#Consultar tamaño de los data frames

print('Tamaño Dataframe ventas: ', df_ventas_giga.shape)
print('Tamaño Dataframe envíos: ',df_envios_giga.shape)

Tamaño Dataframe ventas:  (2500, 6)
Tamaño Dataframe envíos:  (2250, 3)


In [15]:
#Unir Dataframes
df_unido = df_ventas_giga.merge(df_envios_giga, on=['ID_Pedido'], how='left')
print('Tamaño Dataframe envíos: ',df_unido.shape)

#Visualizar nuevo Dataframe
df_unido.head(5)

Tamaño Dataframe envíos:  (2500, 8)


,ID_Pedido,Cliente,Plan_Suscripcion,Compras_2026,Total_Gastado_USD,Region,Dias_Entrega,Estado_Envio
0,10001,QuantumSoft,Familiar,20.0,1177.870664,NaN,4.0,Entregado
1,10002,GlobalNet,Basico,22.0,1788.694792,Sur,5.0,Entregado
2,10003,DeltaSystems,Basico,16.0,2460.093379,Sur,8.0,Retrasado
3,10004,QuantumSoft,Premium,23.0,172.929608,NaN,14.0,Entregado
4,10005,BetaOmega,Premium,11.0,286.901671,Norte,12.0,Retrasado


## 🔍 Desafío 5 (Nivel: Medio) - Reporte Gerencial y Formateo de Métricas
* **Contexto:** El Director Comercial necesita presentar los resultados de facturación a la junta directiva esta tarde. Exige un reporte resumido por región para evaluar el rendimiento geográfico, pero nos pide que los promedios financieros no muestren decenas de decimales para facilitar la lectura rápida.
* **Objetivo:** Tomar el DataFrame unificado (`df_unido`), agrupar los datos por la columna `Region` y calcular dos métricas: la suma total de `Total_Gastado_USD` y el promedio de `Total_Gastado_USD`. El DataFrame resultante debe llamarse `df_reporte_regional` y debe incluir el método `.reset_index()` para quedar en formato plano.
* **Pista conceptual:** Agrupa por la variable geográfica y aplica el método de agregación correspondiente usando un diccionario para calcular ambas métricas sobre la misma columna financiera.


In [16]:
#Realizar el agrupamiento, los cálculos y asignar a un nuevo dataframe
df_reporte_regional = df_unido.groupby('Region')['Total_Gastado_USD'].agg(['sum','mean']).reset_index().round(2)

#Mostrar Dataframe
df_reporte_regional

,Region,sum,mean
0,Este,619358.85,1236.25
1,Norte,957702.44,1251.90
2,Oeste,610667.46,1277.55
3,Sur,677484.60,1271.08


## 🌋 Desafío 6 (Nivel: Difícil) - Reporte Multivariable Cruzado con Filtro de Eficiencia
* **Contexto:** La junta directiva quiere evaluar la eficiencia operativa de la logística de envíos. Nos piden un reporte enfocado exclusivamente en las entregas que ya concluyeron y que sufrieron problemas, es decir, pedidos con estado `"Retrasado"`. Necesitan ver cuántos días promedio tarda un paquete rezagado en llegar y cuánto dinero representan esas demoras, segmentado por el tipo de suscripción del cliente.
* **Objetivo:** Tomar el DataFrame unificado (`df_unido`), filtrar la tabla para conservar únicamente las filas donde el `Estado_Envio` sea exactamente `"Retrasado"`. Luego, agrupar por `Plan_Suscripcion` y calcular simultáneamente el promedio de `Dias_Entrega` y la suma de `Total_Gastado_USD`. Guarda el reporte final en un DataFrame plano llamado `df_auditoria_retrasos` redondeado a 1 decimal.
* **Pista conceptual (Oculta en Spoiler - Haz clic para expandir):**
  <details>
    <summary>💡 Ver pista paso a paso</summary>
    1. Aplica un filtro convencional para quedarte solo con los registros de la categoría de retraso.<br>
    2. Al agrupar el DataFrame resultante por el plan de suscripción, pasa un diccionario a `.agg()` donde asocies a la columna de días la función de promedio, y a la columna financiera la función de suma.<br>
    3. Finaliza la cadena con el formateo de índice y redondeo aprendidos anteriormente.
  </details>


In [22]:
#Ralizar filtro, luego agrupacion y luego a cada columna numerica pasarle la operación en un diccionario
df_unido[df_unido['Estado_Envio'] == 'Retrasado'].groupby('Plan_Suscripcion')[['Dias_Entrega' , 'Total_Gastado_USD']].agg({'Dias_Entrega':'mean' , 'Total_Gastado_USD':'sum'}).reset_index().round(2)

#Llevar esto a un nuevo dataframe df_auditoria_retrasos
df_auditoria_retrasos = df_unido[df_unido['Estado_Envio'] == 'Retrasado'].groupby('Plan_Suscripcion')[['Dias_Entrega' , 'Total_Gastado_USD']].agg({'Dias_Entrega':'mean' , 'Total_Gastado_USD':'sum'}).reset_index().round(2).copy()

#Imprimir dataframe con el reporte
df_auditoria_retrasos

,Plan_Suscripcion,Dias_Entrega,Total_Gastado_USD
0,Basico,7.94,230044.8
1,Familiar,7.51,121051.5
2,Premium,7.16,88721.1
